# CXR Intelligence — Full Pipeline (Google Colab, free T4)

**DSAI 413 Assignment 2** — Colab alternative to the Kaggle notebook (`00_kaggle_full_pipeline.ipynb`).

**Trade-offs vs Kaggle:**
- ✓ Familiar Colab UI, Drive persistence
- ✗ Kaggle dataset must be downloaded (~16.5 GB, ~5 min on Colab) — Kaggle notebook attaches it instantly
- ✗ Colab free tier has tighter GPU quotas and shorter sessions than Kaggle's 30 hr/week

**Use this notebook if:**
- You don't have a Kaggle account
- You want to use Drive for persistence across multiple sessions

## Prerequisites
1. **Runtime → Change runtime type → GPU T4**
2. **Secrets** (🔑 key icon in left sidebar) — add these (toggle Notebook access ON):
   - `HF_TOKEN` (accept [MedGemma TOS](https://huggingface.co/google/medgemma-4b-it) first)
   - `NVIDIA_API_KEY` (free at https://build.nvidia.com)
   - `KAGGLE_USERNAME`, `KAGGLE_KEY` (from https://www.kaggle.com/settings)

## Stage 1 — Bootstrap (Drive mount, repo clone, deps, secrets)

In [ ]:
# Mount Drive for persistent storage
from google.colab import drive, userdata
drive.mount('/content/drive')

# Clone repo (idempotent)
import os, pathlib, subprocess
REPO = '/content/cxr-intel'
if not pathlib.Path(REPO).exists():
    subprocess.run(['git', 'clone', 'https://github.com/jo3591/assigment2dsai413', REPO], check=True)
%cd /content/cxr-intel
!git pull

# Install pinned deps
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# Force a transformers version that supports MedGemma's gemma3 architecture
!pip install -q --force-reinstall --no-deps 'transformers>=4.50,<4.53'

# Load Colab Secrets into env
for k in ['HF_TOKEN','NVIDIA_API_KEY','KAGGLE_USERNAME','KAGGLE_KEY']:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

# Set up Kaggle credentials for dataset download
import json, pathlib
kdir = pathlib.Path.home() / '.kaggle'
kdir.mkdir(exist_ok=True)
(kdir / 'kaggle.json').write_text(json.dumps({
    'username': os.environ['KAGGLE_USERNAME'],
    'key': os.environ['KAGGLE_KEY'],
}))
os.chmod(kdir / 'kaggle.json', 0o600)

import torch
print(f'\nGPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## Stage 2 — Data prep (download + preprocess, ~10 min)

Downloads `simhadrisadaram/mimic-cxr-dataset` (~16.5 GB) from Kaggle, then preprocesses reports + CheXpert labels + patient-level splits. The download is cached under `data/raw/` in your Drive — only re-runs if the cache is missing.

**Note:** On Colab free tier this can be slow; if you've already run this on Kaggle, copy the artifacts from your Quick-Saved version's Output to `/content/drive/MyDrive/cxr_intel_artifacts/` and skip Stages 2–4.

In [ ]:
# Optionally restore preprocessed artifacts from Drive (if a previous session backed them up)
drive_artifacts = pathlib.Path('/content/drive/MyDrive/cxr_intel_artifacts')
if drive_artifacts.exists():
    !cp -r {drive_artifacts}/processed data/ 2>/dev/null || true
    !cp -r {drive_artifacts}/qa data/ 2>/dev/null || true
    !cp -r {drive_artifacts}/indices data/ 2>/dev/null || true
    print('Restored from Drive:', os.listdir('data'))

# Download Kaggle subset if not already present
if not pathlib.Path('data/raw').exists():
    !python -u scripts/download_data.py --config configs/data.yaml --no-basename-index
else:
    # Just (re)preprocess, no re-download
    !python -u scripts/download_data.py --config configs/data.yaml --skip-download --no-basename-index

In [ ]:
# Backup to Drive so you don't lose the preprocessing on a kernel restart
drive_artifacts.mkdir(parents=True, exist_ok=True)
!cp -r data/processed {drive_artifacts}/ 2>/dev/null
print('✓ Backed up data/processed to Drive')

## Stage 3 — Synthesize QA dataset (~90 min, NVIDIA NIM)

In [ ]:
!python -u scripts/build_qa_dataset.py --config configs/data.yaml
!cp -r data/qa {drive_artifacts}/

## Stage 4 — Build retrieval indices (~30 min on T4)

### 4a. BiomedCLIP (5 min)

In [ ]:
!python -u scripts/build_indices.py --backend biomedclip

### 4b. Patch ColPali v1.3 adapter for transformers 5.x

Newer transformers refactored PaliGemma layer paths. We remap the official adapter so it actually merges into the model.

In [ ]:
import shutil, safetensors.torch
from huggingface_hub import snapshot_download

full_src = pathlib.Path(snapshot_download('vidore/colpali-v1.3'))
patched = pathlib.Path('/content/colpali-v1.3-patched')
if patched.exists():
    shutil.rmtree(patched)
patched.mkdir(parents=True)
for src in full_src.iterdir():
    if src.is_file():
        shutil.copy2(src, patched / src.name)

state = safetensors.torch.load_file(str(patched / 'adapter_model.safetensors'))
remapped = {k.replace('model.language_model.model.', 'model.model.language_model.'): v for k, v in state.items()}
safetensors.torch.save_file(remapped, str(patched / 'adapter_model.safetensors'))
print(f'✓ Patched {len(remapped)} adapter keys → {patched}')

### 4c. Build ColPali index using patched checkpoint (~25 min)

In [ ]:
import pandas as pd
from cxr_intel.retrieval.colpali_index import ColPaliRetriever

df = pd.read_parquet('data/processed/reports.parquet')
items = [
    {'study_id': str(r['study_id']),
     'image_path': str(r['image_path']),
     'report_text': (str(r.get('findings','')) + ' ' + str(r.get('impression',''))).strip()}
    for _, r in df.iterrows()
]
r = ColPaliRetriever(
    checkpoint='/content/colpali-v1.3-patched',
    torch_dtype='float16',
    device_map='auto',
    batch_size=2,
)
r.index(items, 'data/indices/colpali_zs')
!cp -r data/indices {drive_artifacts}/

## Stage 5 — Run evaluation

**Restart the runtime** (Runtime → Restart) between Stage 4 and Stage 5 to free GPU memory held by the ColPali model used for indexing. Re-run Stage 1 to restore env, then continue here.

In [ ]:
# Install INT4 quantization support and apply runtime patches
!pip install -q --upgrade 'bitsandbytes>=0.45,<0.50'

# Patch run_eval.py to use INT4 MedGemma + the patched ColPali checkpoint
p = pathlib.Path('scripts/run_eval.py')
content = p.read_text()
new = content.replace(
    'medgemma = MedGemmaRunner()\n    medgemma.load()',
    "medgemma = MedGemmaRunner(quantization='int4')\n    medgemma.load()"
).replace(
    'r = ColPaliRetriever(name="colpali_zs")',
    'r = ColPaliRetriever(name="colpali_zs", checkpoint="/content/colpali-v1.3-patched")',
)
p.write_text(new)
print('✓ Patches applied')

In [ ]:
# Full eval — all 3 modes, 50 test cases per mode (~80 min on T4)
!python -u scripts/run_eval.py \
    --mode report --mode qa --mode retrieval \
    --configs medgemma_only,biomedclip_rag,colpali_zs_rag \
    --test-size 50 --top-k 3

!cp -r results {drive_artifacts}/

In [ ]:
# Inspect final results
import pandas as pd
for name in ['report_metrics', 'qa_metrics', 'retrieval_metrics']:
    print(f'\n=== {name} ===')
    print(pd.read_csv(f'results/tables/{name}.csv').to_string(index=False))

## Stage 6 — Streamlit demo (optional, local)

The Streamlit app under `src/cxr_intel/app/streamlit_app.py` works locally. For a clickable public URL on Colab, see `notebooks/07_gradio_demo_kaggle.ipynb` (adapt the Kaggle Secrets to `userdata.get(...)` for Colab).